In [1]:
print("ok")

ok


In [2]:
%pwd

'c:\\Users\\Sauradeep\\Gen-AI\\Medibot\\research'

In [3]:
import os 
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Sauradeep\\Gen-AI\\Medibot'

In [5]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\Sauradeep\Gen-AI\Medibot\medibot_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Extract Data from the PDF File
def load_pdf_file(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader )
    documents = loader.load() 
    
    return documents

In [7]:
extracted_data = load_pdf_file(data='Data/')

In [8]:
len(extracted_data) 

637

In [ ]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [52]:
text_chunks = text_split(extracted_data)
print(len(text_chunks))

5859


In [14]:
#text_chunks 

In [ ]:
#Download the Embeddings from HuggingFace
from langchain_community.embeddings import HuggingFaceEmbeddings


def download_hugging_face_embedding():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [54]:
embeddings  = download_hugging_face_embedding()

In [55]:
query_result = embeddings.embed_query("Hello World")
print("Length", len(query_result))

Length 384


In [56]:
query_result

[-0.034477293491363525,
 0.031023185700178146,
 0.006734919268637896,
 0.026108955964446068,
 -0.03936200588941574,
 -0.16030244529247284,
 0.06692398339509964,
 -0.00644145580008626,
 -0.047450482845306396,
 0.014758873730897903,
 0.07087531685829163,
 0.05552761256694794,
 0.019193336367607117,
 -0.026251327246427536,
 -0.010109526105225086,
 -0.026940451934933662,
 0.02230745740234852,
 -0.02222668007016182,
 -0.14969263970851898,
 -0.017492998391389847,
 0.007676251698285341,
 0.05435226485133171,
 0.003254401497542858,
 0.031725890934467316,
 -0.08462139964103699,
 -0.029405971989035606,
 0.051595598459243774,
 0.04812406003475189,
 -0.003314854810014367,
 -0.05827920511364937,
 0.04196922481060028,
 0.022210687398910522,
 0.1281888335943222,
 -0.02233893983066082,
 -0.011656275019049644,
 0.06292839348316193,
 -0.032876357436180115,
 -0.0912260189652443,
 -0.03117534890770912,
 0.05269956961274147,
 0.04703487083315849,
 -0.08420306444168091,
 -0.030056191608309746,
 -0.020744830

In [19]:
from dotenv import load_dotenv 
load_dotenv()

True

In [20]:
Pinecone_API_key = os.environ.get('Pinecone_API_key')
Groq_API_key = os.environ.get('Groq_API_key')

In [57]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=Pinecone_API_key)

index_name = "medibot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,   
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [58]:
import os
os.environ["Pinecone_API_key"] = Pinecone_API_key
os.environ["Groq_API_key"] = Groq_API_key

In [23]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
        documents=text_chunks,
        index_name = index_name,
        embedding=embeddings
        
)

In [59]:
docsearch

In [71]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [72]:
retrieved_docs = retriever.invoke("What is acne?")
retrieved_docs

[Document(id='a70e01e1-7452-4a37-b6c6-101a1c5256fe', metadata={'source': 'Data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='02766e09-e4c9-4f45-9ee4-81e979d4e5e8', metadata={'source': 'Data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='069eac9d-b281-4599-8301-631db94b0056', metadata={'source': 'Data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26')]

In [73]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=Groq_API_key,
    model_name="llama-3.1-8b-instant"
)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt
system_prompt = """
You are a medical assistant.

Answer the question ONLY using the provided context.
If the answer is not in the context, say:
"I cannot find the answer in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=system_prompt,
    input_variables=["context", "question"]
)



# Build retrieval chain (clean LCEL version)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [76]:
response = rag_chain.invoke("What is acne?")
print(response)

Acne is a skin condition characterized by the presence of blackheads, whiteheads, and pimples. It is a common disorder that affects many people, particularly during puberty. The exact cause of acne is not known, but it is thought to be related to the combination of excess oil production, bacteria, and hormonal changes.


In [33]:
len(text_chunks)

5859

In [35]:
docs = retriever.invoke("What is acne?")
print(docs)

[Document(id='02766e09-e4c9-4f45-9ee4-81e979d4e5e8', metadata={'source': 'Data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'), Document(id='a70e01e1-7452-4a37-b6c6-101a1c5256fe', metadata={'source': 'Data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'), Document(id='069eac9d-b281-4599-8301-631db94b0056', metadata={'source': 'Data\\Gale Encyclopedia of Medicine Vol. 1 (A-B).pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26')]
